In [2]:
import openai
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd

In [3]:
client = OpenAI()
load_dotenv()
openai.api_key = os.environ.get("OPENAI_API_KEY")

In [4]:
def generate_response(system_prompt, user_prompt):
    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
          messages=[
              {"role": "system", "content": system_prompt},
              {"role": "user", "content": user_prompt}
              ],
              max_tokens=150,
              n=1,
              stop=None,
              temperature=0.5,
    )
    return response.choices[0].message.content.strip()

In [5]:
system_prompt = "You are an assistant that extracts features users love from app reviews."
user_prompt = "Hello!"

generate_response(system_prompt, user_prompt)

'Hello! How can I assist you today?'

In [5]:
df = pd.read_csv('spotify_reviews_post-2023.csv')
df.sample(100)
all_reviews = " ".join(df['content'].tolist())

In [13]:
len(all_reviews)

11770761

In [6]:
# Create the Assistant
assistant = client.beta.assistants.create(
    name="Spotify Review Analyzer",
    instructions="You are a helpful assistant analyzing Spotify app reviews. Be precise and focus on specific features which are highlighted in the reviews.",
    model="gpt-3.5-turbo",
    temperature=0.5
)

# Create a Thread
thread = openai.beta.threads.create()

# Function to add Review to the thread
def add_user_review(review_text):
    openai.beta.threads.messages.create(
        thread_id=thread.id,
        role="user",
        content=review_text
    )

# Function to run the assistant on the thread
def run_assistant():
    run = openai.beta.threads.runs.create(
        thread_id=thread.id,
        assistant_id=assistant.id
    )
    return run

# Function to retrieve and process assistant's response
def get_assistant_response(run_id):
    messages = openai.beta.threads.messages.list(
        thread_id=thread.id
    )
    response_message = messages.data[0]
    return response_message.content[0].text.value

# Example Usage:
reviews = df['content'].sample(10)

for review in reviews:
    add_user_review(review)
    run = run_assistant()
    while run.status != "completed":
        run = openai.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
    assistant_response = get_assistant_response(run.id)
    print(assistant_response)

Issue 1: Login Error
- Some users are experiencing difficulties logging in, receiving an error message stating, "something went wrong."

Issue 2: Ads
- Users find it bothersome that there are unskippable ads within the app.

Issue 3: Song Skipping Limit
- Users are frustrated by the inability to skip songs after reaching the limit of 6 skips.

Issue 4: Inability to Skip Forward
- Users find it inconvenient that they cannot skip forward within songs.
Issue: Enhanced Songs Limitation
- Users who have enhanced their songs on Spotify are disappointed to find out that these enhanced versions cannot be played offline. This limitation was not made clear before enhancing the songs.

Recommendation:
- Users express frustration and a potential loss of membership due to this issue.
- Users are forced to consider redownloading songs in low quality to be able to play them offline.
- Users suggest the need for an option to convert all songs back to low quality easily.
- The issue of not being able t

In [7]:
# Create the Assistant
assistant = client.beta.assistants.create(
    name="Spotify Review Analyzer",
    instructions="You are a helpful assistant analyzing Spotify app reviews. Be precise and focus on specific features which are highlighted in the reviews.",
    model="gpt-3.5-turbo",
    tools=[{"type": "file_search"}],
    temperature=0.5
)

# Create a vector store caled "Financial Statements"
vector_store = client.beta.vector_stores.create(name="spotify_reviews_post-2023_1000")

# Ready the files for upload to OpenAI
file_paths = ["spotify_reviews_post-2023_1000.json"]
file_streams = [open(path, "rb") for path in file_paths]

# Use the upload and poll SDK helper to upload the files, add them to the vector store,
# and poll the status of the file batch for completion.
file_batch = client.beta.vector_stores.file_batches.upload_and_poll(
  vector_store_id=vector_store.id, files=file_streams
)

# You can print the status and the file counts of the batch to see the result of this operation.
print(file_batch.status)
print(file_batch.file_counts)

completed
FileCounts(cancelled=0, completed=1, failed=0, in_progress=0, total=1)


In [8]:
# Update the assistant with the vector store
assistant = client.beta.assistants.update(
  assistant_id=assistant.id,
  tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}},
)

In [9]:
# Create a Thread
thread = openai.beta.threads.create()

# Function to add Review to the thread
def add_user_review(review_text):
    openai.beta.threads.messages.create(
        thread_id=thread.id,
        role="user",
        content=review_text
    )

# Function to run the assistant on the thread
def run_assistant():
    run = openai.beta.threads.runs.create(
        thread_id=thread.id,
        assistant_id=assistant.id
    )
    return run

# Function to retrieve and process assistant's response
def get_assistant_response(run_id):
    messages = openai.beta.threads.messages.list(
        thread_id=thread.id
    )
    response_message = messages.data[0]
    return response_message.content[0].text.value

# Example Usage:
reviews = df['content'].sample(2)

for review in reviews:
    add_user_review(review)
    run = run_assistant()
    while run.status != "completed":
        run = openai.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)
    assistant_response = get_assistant_response(run.id)
    print(assistant_response)

Several users have expressed dissatisfaction with the Spotify app due to various issues, including the requirement to have a premium subscription for basic features like turning off shuffle, peeking into the next song, or making a queue. Users have also reported an increase in ads, glitches, and bugs, as well as feeling pressured to buy premium despite the app's instability. One user specifically mentioned that the "30 minutes" of ad-free music promised by Spotify only lasted about 6 minutes, with the company attributing this discrepancy to an "error" being "worked on"【4:0†source】【4:1†source】.
Users have expressed frustration with Spotify, mentioning issues such as the app becoming worse over time, the heavy emphasis on premium features, and the abundance of ads. Some users have recommended switching to other music apps like Wynk Music due to their dissatisfaction with Spotify's performance and features【8:0†source】【8:3†source】.
